# Simple data-to-figure example

Use the `data-conduit-refactor` kernel. Run the four sections in order; restart the kernel after changing shared Python code.

## 1. Define the DataStructure and load the streams

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from data_conduit.datastructures import slice_stream, slice_stream_for_trial
from data_conduit.integrations.DLC.pose import pose_to_movement
from movement_figures.data_template.loading import build_datastructure

# Select the recording to read. Change these paths/selectors for another dataset.
ROOT = Path("/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Training")
SESSION = "2026-06-21T093349Z"
datastructure = build_datastructure(
    ROOT,
    level_names=("mouseID", "day"),
    l0_selector="MR_M01569515",
    l1_selector="Day21",
    include=(SESSION,),
    streams=("trials", "events", "dlc"),
)
streams = datastructure.load()

## 2. Slice the inputs for this figure

In [ ]:
# Select a recording before using its acquired clock.
position = slice_stream(streams["dlc:position"], selectors={"session": SESSION})
confidence = slice_stream(streams["dlc:confidence"], selectors={"session": SESSION})
pose = pose_to_movement(position, confidence)

# Choose the original trial number. The row supplies its time bounds and flags.
trials = slice_stream(streams["trials"], selectors={"session": SESSION, "trial_index": 1})
trial = trials.iloc[0]
trial_pose = slice_stream_for_trial(pose, trial, time_coord="time")
x_position = slice_stream(
    trial_pose.position,
    selectors={"individual": "individual_0", "keypoint": "body", "space": "x"},
).squeeze(drop=True)

## 3. Define the plotting function

In [ ]:
def plot_position(time, x_position, *, title):
    """Plot the supplied x coordinates on their acquired clock.

    Parameters
    ----------
    time : one-dimensional array-like
        Acquired timestamps in seconds for the selected trial.
    x_position : one-dimensional array-like
        Matching body x coordinates in pixels. Missing values remain gaps.
    title : str
        Title identifying the recording and selected trial.

    Returns
    -------
    matplotlib.figure.Figure
        Figure containing the position trace; the calling cell displays it.
    """
    # Create the axes and draw the already-selected samples.
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(time, x_position, color="tab:blue", linewidth=1)
    ax.set(xlabel="Recording time (s)", ylabel="Body x position (px)", title=title)
    fig.tight_layout()
    return fig

## 4. Return the figure

In [ ]:
position_figure = plot_position(
    x_position.time, x_position, title=f"{SESSION} · trial {trial.trial_index}",
)
plt.close(position_figure)  # Display the returned figure once below.
position_figure
